# Baseline Experiment - Binary Classification - Llama-3.2-3B

Apply Llama-3.2-3B to classify whether a tweet is relevant to a disater or not, in other word, binary classification.

## A. Setup

In [12]:
%pip install transformers tqdm datasets accelerate huggingface_hub python-dotenv peft trl bitsandbytes

Note: you may need to restart the kernel to use updated packages.


In [13]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from dotenv import load_dotenv
load_dotenv()

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorWithPadding
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
import evaluate

import configuration
from src import setup, data_utils, hf_utils
# from src.models import bert


In [14]:
from huggingface_hub import login
login(os.getenv("HF_TOKEN"))

base_model = "meta-llama/Llama-3.2-3B"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## B. Data

### B.1. Load sets

In [15]:
data_frac = data_utils.DATA_FRACTION

df_train, df_val, df_test = data_utils.load_datasets()

### B.2. Shrink dataset size for development purpose

In [16]:
# Comment out this cell to use the full dataset. This is just for quick testing.
train_size = 100
val_size = int(train_size * len(df_val) / len(df_train))
test_size = int(train_size * len(df_test) / len(df_train))

df_train = df_train.sample(n=train_size, random_state=setup.RANDOM_SEED)
df_val = df_val.sample(n=val_size, random_state=setup.RANDOM_SEED)
df_test = df_test.sample(n=test_size, random_state=setup.RANDOM_SEED)

In [17]:
# Load data as Hugging Face Datasets
ds_train, ds_val, ds_test = hf_utils.create_datasets(df_train, df_val, df_test)

## C. Tokenization

In [18]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
# Llama 3.2 does not have a native padding token, which is required for sequence classification batching.
# We must set the padding token to the EOS token and apply this to both the tokenizer and the model configuration.
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

### C.1. Quick preview on dataset to find optimal parameter for `tokenizer`

In [19]:
hf_utils.max_length_dist(df_train, 'tweet_text', tokenizer)


90th percentile: 48.4
95th percentile: 50.199999999999996
99th percentile: 51.64
Absolute Maximum length: 52


So, set the `max_length` to `64`.

### C.2. Do the Tokenization

In [20]:
save_path = Path(f"../tokens/LLama_3_2_3B/{data_frac}")
train_tokenized, val_tokenized, test_tokenized = hf_utils.load_or_tokenize(
    ds_train, ds_val, ds_test, tokenizer, save_path
    , force_retokenize=True
)

Tokenizing datasets...


Map: 100%|██████████| 21/21 [00:00<00:00, 5245.07 examples/s]


Saving tokenized datasets to ../tokens/LLama_3_2_3B/0.1...


Saving the dataset (1/1 shards): 100%|██████████| 21/21 [00:00<00:00, 3809.21 examples/s]


## B. Fine-tuning BERT
### B.1. Model

In [21]:
# Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# model with Classification Head
llama_base = AutoModelForSequenceClassification.from_pretrained(
    base_model, num_labels=2, quantization_config=bnb_config, device_map="auto"
)

# Crucial: Sync the padding token id
llama_base.config.pad_token_id = tokenizer.pad_token_id

ImportError: The installed version of bitsandbytes (<0.43.1) requires CUDA, but CUDA is not available. You may need to install PyTorch with CUDA support or upgrade bitsandbytes to >=0.43.1.

### B.2. Configure LoRA (PEFT)

In [ ]:
# Prepare model for quantized training
llama_base = prepare_model_for_kbit_training(llama_base)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    modules_to_save=["score"] # Instructs PEFT to fully train the classification head
)

peft_model = get_peft_model(llama_base, lora_config)
peft_model.print_trainable_parameters()

### B.2. Fine-tune

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

training_args = TrainingArguments(
    output_dir="./llama-3.2-disaster-classifier",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    optim="paged_adamw_32bit",
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    fp16=True,
    report_to="none"
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

### B.3. Predict on the Test set

In [ ]:
predictions = trainer.predict(test_tokenized)
pred_labels = np.argmax(predictions.predictions, axis=-1)
test_acc = accuracy_metric.compute(predictions=pred_labels, references=test_tokenized["informative"])
test_f1 = f1_metric.compute(predictions=pred_labels, references=test_tokenized["informative"])